<a href="https://colab.research.google.com/github/krishkumarwork3-beep/Deep-Learning-Song-Recognizer/blob/2ns/song_recognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Deep Learning Song Recognizer

This notebook builds a **song identification system** using deep audio embeddings + similarity search — the deep-learning analog of how Shazam works.

**Approach:**
1. Extract deep audio embeddings from a library of reference songs using **OpenL3** (a pretrained self-supervised audio embedding model).
2. Break each song into short overlapping windows and embed each window, so the system is robust to matching short clips.
3. Store all embeddings in a **FAISS** similarity index (fast nearest-neighbor search).
4. Given a query clip (e.g., a noisy 10-second recording), embed it the same way and search the index.
5. Use majority voting across the query's windows to decide which song it came from.

This works even with background noise, partial clips, or re-recordings, since deep embeddings capture perceptual similarity rather than exact waveform matching.

---
**How to use this notebook:**
- Run cells top to bottom.
- Upload your reference song library (the songs you want to be able to recognize) when prompted.
- Upload a query clip (a short recording/snippet) to test recognition.

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch librosa soundfile faiss-cpu

import os
import numpy as np
import librosa
import torch
from transformers import ClapModel, ClapProcessor
import faiss
import pickle
from collections import Counter
from google.colab import files

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "laion/clap-htsat-unfused"

processor = ClapProcessor.from_pretrained(MODEL_ID)
model = ClapModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

CLAP_SR = 48000
EMBEDDING_SIZE = 512

print("Setup complete. Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.7 MB/s eta 0:00:00


preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  614MB            

model.safetensors: downloading bytes:           |  0.00B            

Setup complete. Using device: cuda


## 2. Upload your reference song library

Upload the full songs you want the system to be able to recognize. Supported formats: mp3, wav, m4a, flac (anything `librosa`/`ffmpeg` can decode).

The filename (without extension) will be used as the song's display name — so name your files sensibly, e.g. `Bohemian_Rhapsody_Queen.mp3`.

In [ ]:
os.makedirs("library", exist_ok=True)

print("Upload one or more reference song files:")
uploaded = files.upload()

for fname in uploaded.keys():
    dest = os.path.join("library", fname)
    os.replace(fname, dest)

library_files = [os.path.join("library", f) for f in os.listdir("library")]
print(f"\n{len(library_files)} songs in library:")
for f in library_files:
    print(" -", f)

Upload one or more reference song files:


TypeError: 'NoneType' object is not subscriptable

## 3. Embedding extraction

We use **OpenL3** (`mel256`, `music` content type, 512-d embeddings) with a sliding window (default: 2-second windows, 1-second hop). Each window becomes one vector in our search index, tagged with which song and timestamp it came from.

In [ ]:
SAMPLE_RATE = 48000
HOP_SIZE = 1.0       # seconds between window starts
EMBEDDING_SIZE = 512
CONTENT_TYPE = "music"

def load_audio(path, sr=SAMPLE_RATE):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio

def embed_audio(audio, sr=CLAP_SR, window_sec=5.0, hop_sec=2.5):
    window = int(window_sec * sr)
    hop = int(hop_sec * sr)
    if len(audio) < window:
        audio = np.pad(audio, (0, window - len(audio)))

    chunks, timestamps = [], []
    for start in range(0, len(audio) - window + 1, hop):
        chunks.append(audio[start:start + window])
        timestamps.append(start / sr)
    if not chunks:
        chunks, timestamps = [audio[:window]], [0.0]

    inputs = processor(audios=chunks, sampling_rate=sr, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        emb = model.get_audio_features(**inputs)  # [N, 512]
    return emb.cpu().numpy(), np.array(timestamps)

print("Embedding functions ready.")

## 4. Build the fingerprint database

For every song in the library: load it, embed it in sliding windows, and store the vectors in a FAISS index along with metadata (which song + timestamp each vector belongs to).

In [ ]:
all_embeddings = []
metadata = []  # list of (song_name, timestamp) aligned with all_embeddings rows

for path in library_files:
    song_name = os.path.splitext(os.path.basename(path))[0]
    print(f"Processing: {song_name} ...")
    audio = load_audio(path)
    emb, ts = embed_audio(audio)
    all_embeddings.append(emb)
    metadata.extend([(song_name, float(t)) for t in ts])
    print(f"   -> {emb.shape[0]} embedding windows")

all_embeddings = np.vstack(all_embeddings).astype("float32")
print(f"\nTotal fingerprint vectors: {all_embeddings.shape[0]}")

# Normalize for cosine similarity via inner product
faiss.normalize_L2(all_embeddings)

index = faiss.IndexFlatIP(EMBEDDING_SIZE)
index.add(all_embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")

# Save so you don't have to reprocess the library every session
faiss.write_index(index, "song_index.faiss")
with open("metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Saved index to song_index.faiss and metadata.pkl")

## 5. Recognize a query clip

Upload a short clip (a snippet, a phone recording, a noisy sample — anything a few seconds to ~30s long). The system embeds it the same way, searches for nearest neighbors for each window, and picks the song with the most votes (weighted by similarity).

In [ ]:
def identify_song(query_path, top_k=5, min_similarity=0.75):
    audio = load_audio(query_path)
    emb, ts = embed_audio(audio)
    emb = emb.astype("float32")
    faiss.normalize_L2(emb)

    similarities, indices = index.search(emb, top_k)

    votes = Counter()
    weighted_scores = Counter()

    for window_sims, window_idxs in zip(similarities, indices):
        for sim, idx in zip(window_sims, window_idxs):
            if idx == -1 or sim < min_similarity:
                continue
            song_name, _ = metadata[idx]
            votes[song_name] += 1
            weighted_scores[song_name] += float(sim)

    if not votes:
        return None, []

    ranked = sorted(weighted_scores.items(), key=lambda x: x[1], reverse=True)
    best_song = ranked[0][0]
    return best_song, ranked

print("Upload a query clip to identify:")
query_upload = files.upload()
query_path = list(query_upload.keys())[0]

best_song, ranking = identify_song(query_path)

print("\n--- Results ---")
if best_song is None:
    print("No confident match found. Try a longer clip or lower `min_similarity`.")
else:
    print(f"Best match: {best_song}\n")
    print("Full ranking (song, cumulative weighted similarity):")
    for song, score in ranking:
        print(f"  {song}: {score:.2f}")

## 6. (Optional) Reload a saved index in a future session

Instead of re-embedding your whole library every time, upload `song_index.faiss` and `metadata.pkl` from a previous run and load them directly.

In [ ]:
# Uncomment to reload a previously saved index instead of rebuilding it:

# index = faiss.read_index("song_index.faiss")
# with open("metadata.pkl", "rb") as f:
#     metadata = pickle.load(f)
# print(f"Loaded index with {index.ntotal} vectors covering "
#       f"{len(set(m[0] for m in metadata))} songs.")

## Notes & tuning

- **`min_similarity`** (default 0.75): raise it for stricter matching (fewer false positives), lower it for noisy/short clips.
- **`HOP_SIZE`**: smaller hop = more fingerprint windows per song = better recall but slower indexing and more memory.
- **Scaling up**: `IndexFlatIP` does exact search and is fine for a personal library (hundreds–low thousands of songs). For a large catalog (100k+ songs), switch to an approximate index like `faiss.IndexIVFFlat` or `faiss.IndexHNSWFlat` for speed.
- **Why OpenL3 over training your own CNN**: OpenL3 was self-supervised on millions of videos and gives strong general-purpose audio embeddings out of the box — no labeled training data or GPU training time needed. This is the practical deep-learning approach for a project like this, rather than training a classifier from scratch (which would require you to pre-define a fixed, closed set of song classes).

In [ ]:
"""
Gradio frontend for the deep-learning song recognizer.

Pipeline: OpenL3 audio embeddings + FAISS similarity search.

Install (locally or in Colab):
    pip install gradio openl3 librosa soundfile faiss-cpu tensorflow==2.15.0 numpy==1.26.4

Run:
    python app.py
"""

import os
import pickle
from collections import Counter

import numpy as np
import librosa
import openl3
import faiss
import gradio as gr

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
SAMPLE_RATE = 48000
HOP_SIZE = 1.0            # seconds between window starts
EMBEDDING_SIZE = 512
CONTENT_TYPE = "music"
INDEX_PATH = "song_index.faiss"
METADATA_PATH = "metadata.pkl"

# --------------------------------------------------------------------------
# In-memory state (persisted to disk so it survives app restarts)
# --------------------------------------------------------------------------
state = {"index": None, "metadata": []}  # metadata: list of (song_name, timestamp)


def _load_existing_index():
    if os.path.exists(INDEX_PATH) and os.path.exists(METADATA_PATH):
        state["index"] = faiss.read_index(INDEX_PATH)
        with open(METADATA_PATH, "rb") as f:
            state["metadata"] = pickle.load(f)


_load_existing_index()


# --------------------------------------------------------------------------
# Core audio / embedding helpers
# --------------------------------------------------------------------------
def load_audio(path, sr=SAMPLE_RATE):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio


def embed_audio(audio, sr=SAMPLE_RATE, hop_size=HOP_SIZE):
    emb, ts = openl3.get_audio_embedding(
        audio, sr,
        content_type=CONTENT_TYPE,
        embedding_size=EMBEDDING_SIZE,
        hop_size=hop_size,
        verbose=0,
    )
    return emb, ts


def status_text():
    n_songs = len(set(m[0] for m in state["metadata"]))
    n_vectors = state["index"].ntotal if state["index"] is not None else 0
    if n_vectors == 0:
        return "📭 Library is empty. Upload songs below to build the index."
    return f"📚 Library ready: **{n_songs} songs**, {n_vectors} fingerprint windows indexed."


# --------------------------------------------------------------------------
# Build / extend the library index
# --------------------------------------------------------------------------
def build_library(song_files, progress=gr.Progress()):
    if not song_files:
        return status_text(), gr.update(choices=sorted(set(m[0] for m in state["metadata"])))

    new_embeddings = []
    new_metadata = []

    for i, file_obj in enumerate(song_files):
        path = file_obj.name
        song_name = os.path.splitext(os.path.basename(path))[0]
        progress((i, len(song_files)), desc=f"Embedding {song_name}")

        audio = load_audio(path)
        emb, ts = embed_audio(audio)
        new_embeddings.append(emb.astype("float32"))
        new_metadata.extend([(song_name, float(t)) for t in ts])

    new_embeddings = np.vstack(new_embeddings).astype("float32")
    faiss.normalize_L2(new_embeddings)

    if state["index"] is None:
        state["index"] = faiss.IndexFlatIP(EMBEDDING_SIZE)

    state["index"].add(new_embeddings)
    state["metadata"].extend(new_metadata)

    # Persist to disk
    faiss.write_index(state["index"], INDEX_PATH)
    with open(METADATA_PATH, "wb") as f:
        pickle.dump(state["metadata"], f)

    song_names = sorted(set(m[0] for m in state["metadata"]))
    return status_text(), gr.update(choices=song_names)


def clear_library():
    state["index"] = None
    state["metadata"] = []
    for p in (INDEX_PATH, METADATA_PATH):
        if os.path.exists(p):
            os.remove(p)
    return status_text(), gr.update(choices=[])


# --------------------------------------------------------------------------
# Identify a query clip
# --------------------------------------------------------------------------
def identify_song(query_audio_path, top_k, min_similarity):
    if query_audio_path is None:
        return "Please upload or record a clip first.", None

    if state["index"] is None or state["index"].ntotal == 0:
        return "Library is empty — add reference songs first.", None

    audio = load_audio(query_audio_path)
    emb, ts = embed_audio(audio)
    emb = emb.astype("float32")
    faiss.normalize_L2(emb)

    similarities, indices = state["index"].search(emb, int(top_k))

    votes = Counter()
    weighted_scores = Counter()

    for window_sims, window_idxs in zip(similarities, indices):
        for sim, idx in zip(window_sims, window_idxs):
            if idx == -1 or sim < min_similarity:
                continue
            song_name, _ = state["metadata"][idx]
            votes[song_name] += 1
            weighted_scores[song_name] += float(sim)

    if not weighted_scores:
        return "🤷 No confident match found. Try a longer clip or lower the similarity threshold.", None

    ranked = sorted(weighted_scores.items(), key=lambda x: x[1], reverse=True)
    best_song, best_score = ranked[0]

    max_possible = votes[best_song] * 1.0
    confidence = min(100.0, (best_score / max_possible) * 100) if max_possible else 0.0

    result_md = f"### 🎯 Best match: **{best_song}**\n" \
                f"Confidence: {confidence:.1f}% ({votes[best_song]} matching windows)\n"

    table_rows = [{"Song": song, "Score": round(score, 2), "Matching windows": votes[song]}
                  for song, score in ranked[:10]]

    return result_md, table_rows


# --------------------------------------------------------------------------
# Gradio UI
# --------------------------------------------------------------------------
with gr.Blocks(title="Song Recognizer") as demo:
    gr.Markdown(
        """
        # 🎵 Deep Learning Song Recognizer
        Identify songs from short or noisy audio clips using **OpenL3 embeddings + FAISS similarity search**.
        """
    )

    with gr.Tab("1️⃣ Build library"):
        gr.Markdown("Upload the full songs you want the system to recognize (mp3, wav, m4a, flac).")
        library_status = gr.Markdown(status_text())
        song_upload = gr.File(
            label="Reference songs", file_count="multiple",
            file_types=["audio"]
        )
        with gr.Row():
            build_btn = gr.Button("➕ Add to library", variant="primary")
            clear_btn = gr.Button("🗑️ Clear library", variant="stop")

    with gr.Tab("2️⃣ Identify a clip"):
        gr.Markdown("Upload or record a clip (a few seconds to ~30s) to identify.")
        with gr.Row():
            with gr.Column():
                query_audio = gr.Audio(label="Query clip", type="filepath", sources=["upload", "microphone"])
                top_k_slider = gr.Slider(1, 10, value=5, step=1, label="Neighbors per window (top_k)")
                sim_slider = gr.Slider(0.0, 1.0, value=0.75, step=0.01, label="Minimum similarity threshold")
                identify_btn = gr.Button("🔍 Identify song", variant="primary")
            with gr.Column():
                result_output = gr.Markdown()
                ranking_output = gr.Dataframe(
                    headers=["Song", "Score", "Matching windows"],
                    label="Full ranking"
                )

    known_songs_list = gr.Dropdown(
        label="Songs currently in library", choices=[], interactive=False, visible=True
    )

    build_btn.click(
        fn=build_library,
        inputs=[song_upload],
        outputs=[library_status, known_songs_list],
    )
    clear_btn.click(
        fn=clear_library,
        inputs=[],
        outputs=[library_status, known_songs_list],
    )
    identify_btn.click(
        fn=identify_song,
        inputs=[query_audio, top_k_slider, sim_slider],
        outputs=[result_output, ranking_output],
    )

if __name__ == "__main__":
    demo.launch(debug=True)